# Demo `satuasummarization.extractive`
Peringkasan ekstraktif teks Bali dengan bobot fitur hasil optimasi metaheuristik.

In [ ]:
import sys, json, pickle
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from satuasummarization.extractive import (MetaheuristicSummarizer, SentenceFeatureExtractor, available_models,
                                           FEATURE_NAMES, train_test_split_docs)

## 1. Memakai model pretrained terbaik

In [ ]:
teks = (ROOT / 'datasets/Balinese BATS/1.txt').read_bytes().decode('cp1252')
model = MetaheuristicSummarizer.from_pretrained('best')
out = model.summarize(teks, title='', ratio=0.3)
print('model:', out['algorithm'])
print(out['summary'])

In [ ]:
pd.DataFrame(out['sentences']).head(15)

## 2. Daftar 47 model pretrained dan metrik ujinya

In [ ]:
idx = available_models()
pd.DataFrame(idx['models']).T.drop(columns=['file', 'sha256']).sort_values('test_rouge1_f1', ascending=False).head(15)

## 3. Bobot fitur model terbaik

In [ ]:
w = pd.Series(model.weights, index=FEATURE_NAMES).sort_values()
w.plot.barh(figsize=(7, 5), title=f'Bobot fitur {model.algorithm}'); plt.tight_layout()

## 4. Kurva konvergensi (fitness latih per iterasi, seed terbaik tiap model)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for alg in ['PO', 'PSO', 'GWO', 'SAR', 'PLO', 'SSA']:
    hist = MetaheuristicSummarizer.from_pretrained(alg, extractor=model.extractor).meta['history']
    ax.plot(np.arange(1, len(hist) + 1), hist, label=alg)
ax.set(xlabel='iterasi', ylabel='fitness latih (rerata ROUGE-1/2 F1)', title='Konvergensi pelatihan'); ax.legend(); ax.grid(alpha=.3)

## 5. Melatih ulang satu algoritme dari fitur yang sudah diekstraksi

In [ ]:
docs = pickle.load(open(ROOT / 'results/corpus_features.pkl', 'rb'))
train, test = train_test_split_docs(docs, test_size=0.2, seed=42)
m = MetaheuristicSummarizer(ratio=0.4).fit(train, algorithm='GWO', n_agents=20, max_iter=20, seed=0)
pd.Series(m.evaluate(test)).round(4)